In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
import pandas_ta as ta
import warnings
warnings.filterwarnings('ignore')

# Cell 2: Load Original Data
df = pd.read_csv('EURUSD_clean.csv', index_col=0, parse_dates=True)
print(f"Original data shape: {df.shape}")

# Cell 3: Technical Indicators - Trend
print("Calculating Trend Indicators...")
# Moving Averages
df['SMA_10'] = ta.sma(df['Close'], length=10)
df['SMA_20'] = ta.sma(df['Close'], length=20)
df['SMA_50'] = ta.sma(df['Close'], length=50)
df['EMA_12'] = ta.ema(df['Close'], length=12)
df['EMA_26'] = ta.ema(df['Close'], length=26)

# MACD
macd = ta.macd(df['Close'])
df['MACD'] = macd['MACD_12_26_9']
df['MACD_signal'] = macd['MACDs_12_26_9']
df['MACD_histogram'] = macd['MACDh_12_26_9']

# Cell 4: Technical Indicators - Momentum
print("Calculating Momentum Indicators...")
# RSI
df['RSI_14'] = ta.rsi(df['Close'], length=14)
df['RSI_21'] = ta.rsi(df['Close'], length=21)

# Stochastic
stoch = ta.stoch(df['High'], df['Low'], df['Close'])
df['Stoch_K'] = stoch['STOCHk_14_3_3']
df['Stoch_D'] = stoch['STOCHd_14_3_3']

# Williams %R
df['Williams_R'] = ta.willr(df['High'], df['Low'], df['Close'], length=14)

# Cell 5: Technical Indicators - Volatility
print("Calculating Volatility Indicators...")
# Bollinger Bands
bbands = ta.bbands(df['Close'], length=20, std=2)
df['BB_upper'] = bbands['BBU_20_2.0']
df['BB_middle'] = bbands['BBM_20_2.0']
df['BB_lower'] = bbands['BBL_20_2.0']
df['BB_width'] = (df['BB_upper'] - df['BB_lower']) / df['BB_middle']
df['BB_position'] = (df['Close'] - df['BB_lower']) / (df['BB_upper'] - df['BB_lower'])

# Average True Range (ATR)
df['ATR_14'] = ta.atr(df['High'], df['Low'], df['Close'], length=14)

# Cell 6: Technical Indicators - Volume
print("Calculating Volume Indicators...")
# On-Balance Volume
df['OBV'] = ta.obv(df['Close'], df['Volume'])

# Volume Price Trend
df['VPT'] = ta.vpt(df['Close'], df['Volume'])

# Cell 7: Advanced Indicators
print("Calculating Advanced Indicators...")
# Ichimoku Cloud
ichimoku = ta.ichimoku(df['High'], df['Low'])
if isinstance(ichimoku, tuple):
    df['Ichimoku_A'] = ichimoku[0]['ISA_9']
    df['Ichimoku_B'] = ichimoku[0]['ISB_26']

# Parabolic SAR
df['PSAR'] = ta.psar(df['High'], df['Low'])

# Fibonacci Retracement levels (simplified)
def fibonacci_levels(data, window=100):
    recent_high = data['High'].rolling(window).max()
    recent_low = data['Low'].rolling(window).min()
    diff = recent_high - recent_low
    levels = {
        'Fib_0': recent_low,
        'Fib_236': recent_low + diff * 0.236,
        'Fib_382': recent_low + diff * 0.382,
        'Fib_5': recent_low + diff * 0.5,
        'Fib_618': recent_low + diff * 0.618,
        'Fib_786': recent_low + diff * 0.786,
        'Fib_1': recent_high
    }
    return pd.DataFrame(levels)

fib_df = fibonacci_levels(df)
df = pd.concat([df, fib_df], axis=1)

# Cell 8: Price Action Features
print("Calculating Price Action Features...")
# Candlestick patterns
df['Doji'] = ta.cdl_pattern(df['Open'], df['High'], df['Low'], df['Close'], name='doji')
df['Hammer'] = ta.cdl_pattern(df['Open'], df['High'], df['Low'], df['Close'], name='hammer')
df['Engulfing'] = ta.cdl_pattern(df['Open'], df['High'], df['Low'], df['Close'], name='engulfing')

# Support and Resistance levels (simplified)
def support_resistance(data, window=20):
    data['Resistance'] = data['High'].rolling(window).max()
    data['Support'] = data['Low'].rolling(window).min()
    return data

df = support_resistance(df)

# Cell 9: Market Regime Features
print("Calculating Market Regime Features...")
# Trend strength (ADX)
adx = ta.adx(df['High'], df['Low'], df['Close'])
df['ADX'] = adx['ADX_14']
df['DMP'] = adx['DMP_14']
df['DMN'] = adx['DMN_14']

# Market regime classification
df['Volatility_regime'] = pd.cut(df['ATR_14'], bins=3, labels=['Low', 'Medium', 'High'])
df['Trend_regime'] = pd.cut(df['ADX'], bins=[0, 20, 40, 100], labels=['Ranging', 'Trending', 'Strong_Trend'])

# Cell 10: Statistical Features
print("Calculating Statistical Features...")
# Rolling statistics
for window in [5, 10, 20, 50]:
    df[f'Close_mean_{window}'] = df['Close'].rolling(window).mean()
    df[f'Close_std_{window}'] = df['Close'].rolling(window).std()
    df[f'Close_skew_{window}'] = df['Close'].rolling(window).skew()
    df[f'Close_kurt_{window}'] = df['Close'].rolling(window).kurt()

# Z-score of returns
df['Returns'] = df['Close'].pct_change()
df['Returns_zscore'] = (df['Returns'] - df['Returns'].rolling(20).mean()) / df['Returns'].rolling(20).std()

# Cell 11: Correlation Features
print("Calculating Correlation Features...")
# Rolling correlation between price and volume
df['Price_Volume_corr'] = df['Close'].rolling(50).corr(df['Volume'])

# Cell 12: Clean and Save
print("Cleaning and saving...")
# Drop NaN values from indicator calculations
df_clean = df.dropna()

# Remove infinite values
df_clean = df_clean.replace([np.inf, -np.inf], np.nan).dropna()

print(f"Original shape: {df.shape}")
print(f"Clean shape: {df_clean.shape}")
print(f"Total features created: {len(df_clean.columns)}")

# Save features
df_clean.to_csv('features_with_labels.csv')
print("Features saved to 'features_with_labels.csv'")

# Feature summary
print("\n" + "="*50)
print("FEATURE CATEGORIES SUMMARY")
print("="*50)
categories = {
    'Price Data': ['Open', 'High', 'Low', 'Close', 'Volume'],
    'Trend Indicators': ['SMA_10', 'SMA_20', 'SMA_50', 'EMA_12', 'EMA_26', 'MACD', 'MACD_signal', 'MACD_histogram'],
    'Momentum Indicators': ['RSI_14', 'RSI_21', 'Stoch_K', 'Stoch_D', 'Williams_R'],
    'Volatility Indicators': ['BB_upper', 'BB_middle', 'BB_lower', 'BB_width', 'BB_position', 'ATR_14'],
    'Volume Indicators': ['OBV', 'VPT'],
    'Advanced Indicators': ['Ichimoku_A', 'Ichimoku_B', 'PSAR', 'ADX', 'DMP', 'DMN'],
    'Price Action': ['Doji', 'Hammer', 'Engulfing', 'Resistance', 'Support'],
    'Statistical Features': [col for col in df_clean.columns if any(x in col for x in ['mean', 'std', 'skew', 'kurt', 'zscore'])],
    'Fib Levels': [col for col in df_clean.columns if 'Fib' in col]
}

for category, features in categories.items():
    existing = [f for f in features if f in df_clean.columns]
    print(f"\n{category}: {len(existing)} features")
    if len(existing) <= 5:
        print(f"  {existing}")